# Evaluación comparativa del método Chain-Ladder clásico y versiones robustas para estimar el IBNR bajo contaminación controlada

Este cuaderno presenta el experimento principal del trabajo. El objetivo es comparar el desempeño del método Chain-Ladder clásico frente a tres variantes robustas en la estimación del IBNR cuando los triángulos de desarrollo contienen valores atípicos controlados.

La validación previa del simulador se documenta en un cuaderno separado: `validacion_simulador_ibnr.ipynb`. En este archivo se parte de esa validación y se desarrolla el análisis central: definición de escenarios, ejecución Monte Carlo, cálculo de métricas, comparación entre métodos e interpretación de resultados.

## Estructura general del cuaderno

El desarrollo se organiza para seguir el orden del experimento final, sin mezclarlo con la prueba interna del simulador:

1. **Planteamiento del estudio.** Se presentan la pregunta, el objetivo y las hipótesis de trabajo.
2. **Diseño experimental.** Se fijan la dimensión del triángulo, los supuestos de simulación y la matriz de escenarios.
3. **Métodos de estimación.** Se definen el Chain-Ladder clásico y las variantes robustas que serán comparadas.
4. **Experimento Monte Carlo.** Se ejecutan las réplicas bajo los escenarios definidos.
5. **Evaluación de resultados.** Se calculan métricas, rankings y comparaciones pareadas frente al método clásico.
6. **Conclusiones.** Se conectan los resultados con las hipótesis y con la pregunta de investigación.

La prueba del simulador no se elimina del proyecto; queda separada para que funcione como soporte metodológico previo y no interrumpa la lectura del experimento principal.

## Planteamiento del estudio

**Pregunta de investigaci?n.** ?En qu? medida la presencia de valores at?picos altera el desempe?o del m?todo Chain-Ladder cl?sico en la estimaci?n del IBNR y bajo qu? combinaciones de contaminaci?n las variantes robustas ofrecen estimaciones m?s precisas y estables?

**Objetivo general.** Comparar el desempe?o del m?todo Chain-Ladder cl?sico y de tres variantes robustas en la estimaci?n del IBNR, mediante simulaci?n Monte Carlo de tri?ngulos de desarrollo bajo escenarios controlados de contaminaci?n, para establecer si los enfoques robustos ofrecen ventajas en precisi?n y estabilidad cuando la informaci?n observada contiene outliers.

**Hip?tesis de trabajo.**

- H1. En escenarios contaminados, el error y la variabilidad del m?todo Chain-Ladder cl?sico aumentan a medida que crecen la proporci?n y la magnitud de la contaminaci?n, mientras que al menos una variante robusta mantiene un comportamiento relativamente m?s estable.

La hip?tesis se eval?a al final del cuaderno mediante m?tricas de error, desviaci?n est?ndar, rankings por escenario y comparaciones pareadas sobre las mismas r?plicas simuladas.

## Fundamento teórico y criterio de diseño

La literatura actuarial establece que el Chain-Ladder clásico es altamente sensible a observaciones atípicas. Mack (1993) formaliza el marco estocástico del método y muestra cómo cuantificar el error de predicción. England y Verrall (2002) destacan que la principal ventaja de los modelos estocásticos de reserving es la disponibilidad de medidas explícitas de precisión. Verdonck et al. (2009) y Verdonck y Debruyne (2011) muestran, desde la perspectiva de robustez e influencia, que incluso una sola observación extrema puede desplazar de forma importante la reserva estimada. Más recientemente, Avanzi et al. (2024) profundizan en la sensibilidad posicional del reserving frente a outliers y documentan que las celdas cercanas a la frontera observada pueden ejercer una influencia particularmente alta.

A partir de ese marco, el diseño implementado adopta una lógica deliberadamente comparativa: el IBNR real se conoce por construcción, la contaminación se introduce únicamente sobre la región observable y los métodos compiten sobre los mismos triángulos simulados. Esta estructura produce comparaciones pareadas, reduce el ruido Monte Carlo en las diferencias entre métodos y permite una lectura más sólida de la evidencia empírica.

## Ruta metodol?gica del experimento principal

El experimento principal se desarrolla como una secuencia de pasos que conviene leer en orden:

1. Se fijan la dimensi?n del tri?ngulo, la distribuci?n de base y los par?metros del proceso generador.
2. Se construye la matriz de escenarios con base en la proporci?n, la magnitud y la ubicaci?n de la contaminaci?n.
3. Se simulan tri?ngulos incrementales completos y se obtiene su versi?n acumulada.
4. Se define la regi?n observada y se calcula el IBNR verdadero en la zona futura no observada.
5. Se introduce contaminaci?n solo en la regi?n observable del tri?ngulo incremental.
6. Se estiman los factores de desarrollo y el IBNR con el m?todo cl?sico y con las variantes robustas.
7. Se repite el procedimiento para todas las r?plicas y todos los escenarios del dise?o.
8. Se resumen los resultados con m?tricas de error, rankings por escenario y comparaciones pareadas frente al m?todo cl?sico.

La validaci?n del simulador corresponde a una etapa previa y separada. Su funci?n es verificar que la simulaci?n respeta los supuestos del estudio. Por esa raz?n se documenta en `validacion_simulador_ibnr.ipynb` y no dentro de los resultados finales.

## Entorno computacional y reproducibilidad

La implementación se apoya en librerías de uso estándar para simulación, análisis tabular y visualización:

- `numpy`: simulación aleatoria, álgebra matricial y operaciones vectorizadas.
- `pandas`: organización tabular de réplicas, escenarios y métricas.
- `matplotlib` y `seaborn`: visualización de resultados.
- `pathlib`: manejo portable de rutas del repositorio.

El código reutilizable se encuentra en `src/ibnr_project`. Este cuaderno concentra el experimento principal y la discusión de resultados, mientras que la validación del simulador queda documentada por separado. La semilla aleatoria se fija de forma explícita para garantizar reproducibilidad.

In [ ]:
# Preparación de rutas del proyecto
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

OUTPUT_DIR = ROOT / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Librerías y funciones necesarias para el experimento principal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from ibnr_project.config import build_default_config, build_default_scenarios, clone_config
from ibnr_project.diagnostics import (
    build_link_ratio_count_table,
    compute_running_statistics,
    summarize_method_dominance,
)
from ibnr_project.evaluation import (
    compare_methods_to_baseline,
    compute_method_metrics,
    rank_methods_within_scenario,
)
from ibnr_project.experiment import build_global_summary, run_experiment
from ibnr_project.methods import estimate_ibnr_all_methods
from ibnr_project.simulation import observed_mask, simulate_single_triangle

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titleweight"] = "bold"

## Configuración experimental

El experimento se implementa sobre triángulos de dimensión `10 × 10`, donde las filas representan años de ocurrencia y las columnas periodos de desarrollo. Esta dimensión permite trabajar con un horizonte completo de diez años sin producir un costo computacional excesivo para las réplicas Monte Carlo.

Los montos incrementales se generan con esperanza `E[X_{i,j}] = μ_i d_j` y varianza `Var(X_{i,j}) = φ (μ_i d_j)^2`. El patrón acumulado se fija de forma exógena y de él se derivan las proporciones incrementales esperadas.

In [ ]:
# Definición de parámetros base y escenarios del experimento
SEED = 20260503
config = build_default_config(random_seed=SEED, distribution="gamma")
scenarios = build_default_scenarios()
mask = observed_mask(config.n_periods)

scenario_df = pd.DataFrame([scenario.__dict__ for scenario in scenarios])
scenario_df

La tabla anterior contiene la matriz completa de escenarios. El experimento combina tres dimensiones de contaminación:

- **Proporción de celdas contaminadas**: `0%`, `5%`, `10%` y `20%`.
- **Magnitud del outlier**: multiplicación por `1`, `2`, `5` y `10`. El factor `1` solo aparece en el escenario base, donde no hay contaminación efectiva.
- **Ubicación de la contaminación**:
  - `random`: selección aleatoria dentro de la región observable;
  - `early`: primer tercio de celdas observadas de cada fila;
  - `late`: último tercio de celdas observadas de cada fila.

De esta forma, el estudio no se limita a distinguir entre “con” y “sin” outliers, sino que explora cuánto cambia el desempeño del reserving cuando la contaminación es más frecuente, más intensa o aparece cerca de la frontera observada.

## Justificaci?n de decisiones metodol?gicas

Las decisiones centrales del dise?o se resumen a continuaci?n para dejar expl?cito qu? se hizo y cu?l es su papel dentro del estudio.

In [ ]:
# Resumen de las principales decisiones metodol?gicas del experimento
justification_df = pd.DataFrame(
    [
        {
            "decision": "Tri?ngulos 10 ? 10",
            "justificacion": "Es una dimensi?n cl?sica en reserving, suficiente para reproducir una zona observada y una zona futura relevantes sin volver innecesariamente pesado el experimento.",
            "papel_en_el_estudio": "Permite comparar m?todos con una estructura triangular completa y con n?mero decreciente de ratios por periodo.",
        },
        {
            "decision": "1000 r?plicas en el caso base",
            "justificacion": "Se busca estabilizar m?tricas como RMSE, MAPE y desviaci?n est?ndar sin perder viabilidad computacional.",
            "papel_en_el_estudio": "Sostiene la comparaci?n Monte Carlo entre m?todos y escenarios.",
        },
        {
            "decision": "Distribuci?n Gamma como caso base",
            "justificacion": "Genera montos positivos y asim?tricos, compatibles con pagos incrementales, y permite controlar media y dispersi?n de forma directa.",
            "papel_en_el_estudio": "Define el escenario principal del experimento.",
        },
        {
            "decision": "Sensibilidad Lognormal",
            "justificacion": "Permite revisar si los hallazgos cambian cuando la distribuci?n subyacente tiene colas m?s pesadas.",
            "papel_en_el_estudio": "Eval?a la estabilidad de las conclusiones frente a un mecanismo alternativo de simulaci?n.",
        },
        {
            "decision": "Contaminaci?n positiva y multiplicativa",
            "justificacion": "A?sla el efecto de observaciones influyentes de gran magnitud sobre los factores de desarrollo.",
            "papel_en_el_estudio": "Permite medir c?mo se distorsiona el IBNR cuando los datos observados contienen outliers.",
        },
        {
            "decision": "Proporciones 5 %, 10 % y 20 %",
            "justificacion": "Representan niveles bajo, intermedio y alto de contaminaci?n para estudiar sensibilidad de manera gradual.",
            "papel_en_el_estudio": "Distingue entre contaminaci?n escasa, moderada y frecuente.",
        },
        {
            "decision": "Magnitudes ?2, ?5 y ?10",
            "justificacion": "Representan outliers moderados, intensos y severos dentro de una regla de contaminaci?n f?cil de interpretar.",
            "papel_en_el_estudio": "Permite separar el efecto de la frecuencia del efecto de la severidad.",
        },
        {
            "decision": "Media truncada al 10 %",
            "justificacion": "Se adopta como regla robusta moderada, aunque en tri?ngulos peque?os su efecto pr?ctico puede ser limitado por el bajo n?mero de ratios por periodo.",
            "papel_en_el_estudio": "Sirve como variante intermedia entre el promedio cl?sico y reglas m?s resistentes como la mediana.",
        },
    ]
)

justification_df

In [ ]:
# Resumen de las dimensiones usadas para construir los escenarios
scenario_design_df = pd.DataFrame(
    {
        "componente": ["Proporción", "Magnitud", "Ubicación"],
        "niveles": ["0%, 5%, 10%, 20%", "x1, x2, x5, x10", "none, random, early, late"],
        "interpretación": [
            "Determina qué fracción de las celdas observadas es alterada.",
            "Determina cuán extremo se vuelve cada valor atípico seleccionado.",
            "Determina si la contaminación ocurre al inicio, al final o de forma dispersa en el desarrollo.",
        ],
    }
)
scenario_design_df

In [ ]:
# Revisión de los parámetros de desarrollo usados en la simulación
parameter_overview = pd.DataFrame(
    {
        "periodo_desarrollo": np.arange(1, config.n_periods + 1),
        "proporcion_acumulada": config.development_cumulative,
        "proporcion_incremental": config.development_incremental,
    }
)

print("Semilla:", config.random_seed)
print("Distribución base:", config.distribution)
print("Dispersión phi:", config.dispersion_phi)
print("Ultimates esperados por año de ocurrencia:", np.round(config.ultimate_means, 2))
parameter_overview

El modelo base puede resumirse en la forma

$$
\begin{aligned}
X_{i,j} &\sim \mathrm{Gamma}(\alpha_{i,j}, \beta_{i,j}), \\
\mathbb{E}[X_{i,j}] &= \mu_i d_j, \\
\mathrm{Var}(X_{i,j}) &= \phi (\mu_i d_j)^2.
\end{aligned}
$$

Aquí, $\mu_i$ representa el nivel esperado último del año de ocurrencia $i$, $d_j$ representa la fracción incremental esperada del periodo de desarrollo $j$ y $\phi$ controla la dispersión relativa. El patrón acumulado fija la forma general del desarrollo, mientras que los $\mu_i$ introducen heterogeneidad entre filas. Esta combinación permite generar triángulos plausibles sin perder trazabilidad estadística.

## Tamaño muestral de los ratios de desarrollo

Antes de aplicar los métodos, se revisa cuántos ratios individuales quedan disponibles por periodo de desarrollo. Esta información es relevante porque las variantes robustas estiman factores mediante mediana, media truncada o ponderación robusta, y su estabilidad depende del número de ratios disponibles en cada periodo.

In [ ]:
# Conteo de ratios disponibles por periodo de desarrollo
ratio_count_df = build_link_ratio_count_table(mask)
ratio_count_df

In [ ]:
# Visualización del número de ratios usados para estimar factores de desarrollo
plt.figure(figsize=(8, 4))
sns.barplot(data=ratio_count_df, x="development_period", y="n_individual_ratios", color="#35608d")
plt.title("Número de ratios individuales por periodo de desarrollo")
plt.xlabel("Periodo de desarrollo j")
plt.ylabel("Número de ratios C(i,j+1) / C(i,j)")
plt.tight_layout()
plt.show()

## Métodos de estimación

Se comparan cuatro formas de estimar los factores de desarrollo y el IBNR:

- **Chain-Ladder clásico:** usa el promedio ponderado tradicional de los ratios de desarrollo.
- **Chain-Ladder con mediana:** reemplaza el promedio por la mediana de los ratios individuales para reducir el efecto de valores extremos.
- **Chain-Ladder con media truncada:** elimina una proporción de ratios en los extremos antes de calcular el promedio.
- **Chain-Ladder ponderado robusto:** reduce el peso de ratios alejados del comportamiento central mediante una escala robusta.

Todos los métodos se aplican sobre los mismos triángulos observados en cada réplica. Por ello, las diferencias de desempeño se atribuyen al método de estimación y no a cambios en los datos de entrada.

En términos operativos, el procedimiento de estimación sigue tres pasos:

1. Calcular ratios individuales de desarrollo a partir del triángulo acumulado observado.
2. Estimar factores agregados de desarrollo con cada método.
3. Proyectar la parte futura del triángulo y calcular el IBNR estimado.

La siguiente celda muestra una verificación mínima de salida para una réplica ilustrativa. La revisión completa del simulador está en el notebook de validación.

In [ ]:
# Verificación mínima de los métodos sobre una réplica ilustrativa
rng_preview = np.random.default_rng(SEED)
preview_example = simulate_single_triangle(config, scenarios[4], rng_preview)
estimate_preview = estimate_ibnr_all_methods(preview_example.observed_cumulative, config)
estimate_preview

## Diseño Monte Carlo y estructura pareada

El experimento principal usa `1000` réplicas para cada escenario bajo distribución Gamma. En cada réplica, los cuatro métodos se aplican sobre el mismo triángulo observado. Esta estructura genera comparaciones pareadas: cada diferencia entre métodos se calcula con la misma información simulada, lo que reduce el ruido de comparación y permite evaluar directamente el efecto del método de estimación.

In [ ]:
# Ejecución del experimento Monte Carlo principal
N_REPLICAS = 1000
results_df = run_experiment(config, scenarios, n_replicas=N_REPLICAS)
results_df["error"] = results_df["estimated_ibnr"] - results_df["true_ibnr"]
results_df["absolute_error"] = np.abs(results_df["error"])
results_df["percentage_error"] = results_df["error"] / results_df["true_ibnr"]
results_df["absolute_percentage_error"] = np.abs(results_df["percentage_error"])

scenario_meta = (
    results_df.groupby("scenario", as_index=False)
    .agg(
        contamination_proportion=("contamination_proportion", "first"),
        contamination_magnitude=("contamination_magnitude", "first"),
        contamination_location=("contamination_location", "first"),
    )
    .sort_values("scenario")
    .reset_index(drop=True)
)

results_df.head()

## Métricas principales

El desempeño se resume mediante sesgo, MSE, RMSE, MAE, MAPE, error porcentual medio y desviación estándar de las estimaciones. Para las métricas basadas en promedios también se reporta una aproximación al error estándar Monte Carlo y un intervalo de confianza normal aproximado. El RMSE permanece como métrica principal de comparación, por su lectura directa en la escala del IBNR.

Las métricas principales se interpretan del siguiente modo:

$$
\begin{aligned}
\mathrm{Bias} &= \frac{1}{N}\sum_{s=1}^{N}(\widehat{IBNR}_s - IBNR_s), \\
\mathrm{RMSE} &= \sqrt{\frac{1}{N}\sum_{s=1}^{N}(\widehat{IBNR}_s - IBNR_s)^2}.
\end{aligned}
$$

$$
\begin{aligned}
\mathrm{MAPE} &= \frac{1}{N}\sum_{s=1}^{N}\left|\frac{\widehat{IBNR}_s - IBNR_s}{IBNR_s}\right|, \\
\operatorname{SD}(\widehat{IBNR}) &= \sqrt{\frac{1}{N-1}\sum_{s=1}^{N}(\widehat{IBNR}_s - \overline{\widehat{IBNR}})^2}.
\end{aligned}
$$

- Un **RMSE** menor indica mejor precisión global.
- Un **MAPE** menor indica menor error relativo medio.
- Un **sesgo** cercano a cero indica ausencia de sobreestimación o subestimación sistemática.
- Una **desviación estándar** menor indica mayor estabilidad entre réplicas.

Por ello, un método favorable no es necesariamente el que minimiza una sola métrica, sino el que presenta un compromiso razonable entre precisión, estabilidad y comportamiento sistemático.

In [ ]:
# Cálculo de métricas de desempeño por método y escenario
metrics_df = compute_method_metrics(results_df).merge(scenario_meta, on="scenario", how="left")
metrics_df.head(12)

In [ ]:
# Ranking de métodos según RMSE y resumen global de desempeño
ranking_df = rank_methods_within_scenario(metrics_df, metric="rmse")
dominance_df = summarize_method_dominance(ranking_df)
global_summary = build_global_summary(metrics_df)

print("Método ganador por número de escenarios:")
display(dominance_df)
print("Resumen global por método:")
display(global_summary)

Las dos tablas anteriores cumplen funciones distintas. La primera cuenta cuántos escenarios gana cada método cuando el criterio de comparación es el RMSE. La segunda resume su comportamiento promedio en el conjunto total del experimento. La combinación de ambas tablas permite separar dos preguntas distintas: qué método domina globalmente y qué método conserva ventajas en escenarios específicos.

In [ ]:
# Resumen comparativo entre escenario limpio y escenarios contaminados
environment_summary = (
    metrics_df.assign(
        entorno=lambda df: np.where(df["scenario"].eq("base_sin_contaminacion"), "Sin contaminación", "Escenarios contaminados")
    )
    .groupby(["entorno", "method"], as_index=False)
    .agg(
        mean_rmse=("rmse", "mean"),
        mean_mape=("mape", "mean"),
        mean_sd=("sd_estimates", "mean"),
    )
)

best_by_scenario = (
    ranking_df.loc[ranking_df["rank"] == 1, ["scenario", "method", "rmse", "mape", "bias",
                                             "contamination_proportion", "contamination_magnitude",
                                             "contamination_location"]]
    .sort_values(["contamination_proportion", "contamination_magnitude", "contamination_location"])
    .reset_index(drop=True)
)

print("Resumen por entorno:")
display(environment_summary)
print("Método ganador por escenario:")
display(best_by_scenario.head(15))

In [ ]:
# Visualización general del RMSE por método y por escenario
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=global_summary, x="method", y="mean_rmse", hue="method", palette="crest", legend=False, ax=axes[0])
axes[0].set_title("RMSE promedio por método")
axes[0].set_xlabel("Método")
axes[0].set_ylabel("RMSE promedio")

heatmap_data = (
    metrics_df.pivot_table(index="scenario", columns="method", values="rmse")
    .reindex(sorted(metrics_df["scenario"].unique()))
)
sns.heatmap(heatmap_data, cmap="YlGnBu", ax=axes[1], cbar_kws={"label": "RMSE"})
axes[1].set_title("RMSE por escenario y método")
axes[1].set_xlabel("Método")
axes[1].set_ylabel("Escenario")

plt.tight_layout()
plt.show()

La lectura conjunta del gráfico de barras y del mapa de calor permite identificar dos hechos centrales. En primer lugar, el método con mejor desempeño global no siempre coincide con el más eficiente en el escenario base. En segundo lugar, la intensidad de color del mapa de calor permite detectar si el deterioro del método clásico ocurre de forma aislada o si se concentra sistemáticamente en combinaciones de alta proporción y alta magnitud de contaminación.

## Comparaciones pareadas frente al método clásico

La evaluación anterior resume el desempeño medio por escenario. Sin embargo, dado que los métodos se aplican sobre las mismas réplicas, es posible comparar sus errores en forma pareada. Una diferencia negativa en `delta_mse_mean` indica que el método robusto presenta menor error cuadrático medio que el método clásico en el mismo conjunto de réplicas.

In [ ]:
# Comparación pareada de cada método robusto contra Chain-Ladder clásico
comparisons_df = compare_methods_to_baseline(results_df, baseline="classical")
comparisons_df = comparisons_df.merge(scenario_meta, on="scenario", how="left")
comparisons_df.head(12)

In [ ]:
# Resumen de escenarios donde cada método robusto mejora o empeora frente al clásico
comparison_summary = (
    comparisons_df.groupby("method", as_index=False)
    .agg(
        escenarios_con_mejora_mse=("delta_mse_improves", "sum"),
        escenarios_con_deterioro_mse=("delta_mse_worsens", "sum"),
        escenarios_con_mejora_mape=("delta_mape_improves", "sum"),
        delta_mse_promedio=("delta_mse_mean", "mean"),
        delta_mape_promedio=("delta_mape_mean", "mean"),
    )
    .sort_values("escenarios_con_mejora_mse", ascending=False)
)
comparison_summary

En esta tabla, una mejora significa que el método robusto reduce el error respecto del método clásico sobre las mismas réplicas. Por ejemplo, si `escenarios_con_mejora_mse` es alto para la mediana, ello indica que la ventaja observada no se debe solo a uno o dos escenarios extremos, sino a un patrón repetido en distintos contextos de contaminación.

## Sensibilidad por escenario representativo

La combinación de gráficos de distribución y tablas de ranking permite identificar de forma concreta dónde se produce el deterioro del método clásico y bajo qué tipos de contaminación aparecen ventajas robustas.

In [ ]:
# Boxplots separados para evitar saturación visual en los escenarios representativos
selected_scenarios = [
    "base_sin_contaminacion",
    "p5_m2_random",
    "p5_m10_late",
    "p10_m5_random",
    "p10_m10_early",
    "p20_m5_early",
    "p20_m10_random",
]

n_cols = 2
n_rows = int(np.ceil(len(selected_scenarios) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows), sharey=True)
axes = axes.flatten()

for ax, scenario in zip(axes, selected_scenarios):
    sns.boxplot(
        data=results_df.query("scenario == @scenario"),
        x="method",
        y="percentage_error",
        hue="method",
        palette="Set2",
        legend=False,
        ax=ax,
    )
    ax.axhline(0.0, color="black", linestyle="--", linewidth=1)
    ax.set_title(scenario)
    ax.set_xlabel("Método")
    ax.set_ylabel("Error porcentual")

for ax in axes[len(selected_scenarios):]:
    ax.axis("off")

plt.suptitle("Distribución del error porcentual por escenario representativo", y=1.01)
plt.tight_layout()
plt.show()

En este gráfico, la línea horizontal en cero representa estimación sin error. Valores por encima de cero indican sobreestimación y valores por debajo de cero indican subestimación. La dispersión vertical de cada caja refleja estabilidad: cajas más altas o colas más largas indican mayor variabilidad del método. Esta visualización resulta más informativa que comparar niveles absolutos del IBNR, porque normaliza los errores respecto del tamaño real del IBNR en cada escenario.

## Justificación empírica del número de réplicas

El número de réplicas no se adopta únicamente por convención. La trayectoria acumulada del RMSE muestra si la estimación del desempeño se estabiliza conforme aumenta el tamaño de la simulación. La evidencia siguiente se presenta para dos situaciones contrastantes: el escenario base con el método clásico y un escenario severamente contaminado con la mediana.

In [ ]:
# Revisión de estabilidad empírica del número de réplicas usado
running_base = compute_running_statistics(results_df, "base_sin_contaminacion", "classical")
running_severe = compute_running_statistics(results_df, "p20_m10_random", "median")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.lineplot(data=running_base, x="replica", y="running_rmse", color="#1b4965", ax=axes[0])
axes[0].set_title("Convergencia del RMSE acumulado | Escenario base - Método clásico")
axes[0].set_xlabel("Número de réplicas")
axes[0].set_ylabel("RMSE acumulado")

sns.lineplot(data=running_severe, x="replica", y="running_rmse", color="#9c2f2f", ax=axes[1])
axes[1].set_title("Convergencia del RMSE acumulado | Escenario p20_m10_random - Mediana")
axes[1].set_xlabel("Número de réplicas")
axes[1].set_ylabel("RMSE acumulado")

plt.tight_layout()
plt.show()

## Evaluaci?n de hip?tesis

La hip?tesis del estudio se eval?a con tres fuentes de evidencia: el comportamiento del error del m?todo cl?sico, la variabilidad relativa de las variantes robustas y la comparaci?n por escenarios contaminados.

In [ ]:
# Evaluaci?n sint?tica de la hip?tesis con base en m?tricas observadas
classical_metrics = metrics_df.query("scenario != 'base_sin_contaminacion' and method == 'classical'").copy()
classical_metrics["log_magnitude"] = np.log(classical_metrics["contamination_magnitude"])
correlation_table = classical_metrics[
    ["contamination_proportion", "contamination_magnitude", "log_magnitude", "rmse", "mape", "sd_estimates"]
].corr(method="spearman")

support_sd = (
    metrics_df.query("scenario != 'base_sin_contaminacion'")
    .pivot(index="scenario", columns="method", values="sd_estimates")
    .assign(
        median_less_than_classical=lambda df: df["median"] < df["classical"],
        trimmed_less_than_classical=lambda df: df["trimmed"] < df["classical"],
        weighted_less_than_classical=lambda df: df["weighted"] < df["classical"],
    )
)

robust_sd_summary = pd.DataFrame(
    {
        "metodo_robusto": ["median", "trimmed", "weighted"],
        "escenarios_con_menor_sd_que_clasico": [
            int(support_sd["median_less_than_classical"].sum()),
            int(support_sd["trimmed_less_than_classical"].sum()),
            int(support_sd["weighted_less_than_classical"].sum()),
        ],
    }
)
robust_sd_summary["total_escenarios_contaminados"] = len(support_sd)
robust_sd_summary["proporcion_favorable"] = (
    robust_sd_summary["escenarios_con_menor_sd_que_clasico"]
    / robust_sd_summary["total_escenarios_contaminados"]
)

best_stability_row = robust_sd_summary.sort_values("escenarios_con_menor_sd_que_clasico", ascending=False).iloc[0]
best_stability_method = best_stability_row["metodo_robusto"]
best_stability_count = int(best_stability_row["escenarios_con_menor_sd_que_clasico"])

hypothesis_table = pd.DataFrame(
    [
        {
            "hipotesis": "H1",
            "evidencia_principal": (
                f"Correlaci?n de Spearman del RMSE cl?sico con proporci?n = "
                f"{correlation_table.loc['contamination_proportion', 'rmse']:.3f}, "
                f"con magnitud = {correlation_table.loc['contamination_magnitude', 'rmse']:.3f}. "
                f"En variabilidad, el m?todo robusto con mejor resultado es {best_stability_method}, "
                f"que mejora al cl?sico en {best_stability_count} de {len(support_sd)} escenarios contaminados."
            ),
            "dictamen": "Apoyada" if (
                correlation_table.loc["contamination_proportion", "rmse"] > 0
                and correlation_table.loc["contamination_magnitude", "rmse"] > 0
                and best_stability_count >= len(support_sd) / 2
            ) else "Parcialmente apoyada",
        },
    ]
)

print("Correlaciones de Spearman para el m?todo cl?sico en escenarios contaminados:")
display(correlation_table)
print("Resumen de estabilidad relativa de los m?todos robustos:")
display(robust_sd_summary)
print("Evaluaci?n sint?tica de la hip?tesis:")
display(hypothesis_table)

In [ ]:
# Síntesis de hallazgos principales del experimento Gamma
clean_winner = best_by_scenario.loc[best_by_scenario["scenario"] == "base_sin_contaminacion", "method"].iloc[0]
contaminated_wins = (
    best_by_scenario.query("scenario != 'base_sin_contaminacion'")
    .groupby("method")
    .size()
    .to_dict()
)
top_median_advantages = (
    comparisons_df.query("method == 'median'")
    .sort_values("delta_mse_mean")
    .loc[:, ["scenario", "delta_mse_mean", "delta_mape_mean",
             "contamination_proportion", "contamination_magnitude", "contamination_location"]]
    .head(5)
)

key_results_df = pd.DataFrame(
    [
        {"hallazgo": "Método ganador en el escenario base", "resultado": clean_winner},
        {
            "hallazgo": "Escenarios contaminados ganados por la mediana",
            "resultado": contaminated_wins.get("median", 0),
        },
        {
            "hallazgo": "Escenarios contaminados ganados por el método clásico",
            "resultado": contaminated_wins.get("classical", 0),
        },
        {
            "hallazgo": "Método con menor RMSE promedio global",
            "resultado": global_summary.iloc[0]["method"],
        },
    ]
)

print("Hallazgos principales del experimento Gamma:")
display(key_results_df)
print("Escenarios con mayor ventaja de la mediana frente al método clásico:")
display(top_median_advantages)

La evidencia obtenida muestra un patrón robusto. En datos limpios o con contaminación leve, el método clásico conserva ventajas de eficiencia. Sin embargo, conforme aumentan la magnitud y la frecuencia de la contaminación, la mediana se convierte en la opción más estable y precisa. En esta implementación, la media truncada no confirma la hipótesis de equilibrio robustez-eficiencia y la versión ponderada no domina de forma sistemática. Ese resultado no invalida el diseño; por el contrario, aporta una conclusión empírica relevante: no toda robustificación heurística resulta efectiva en triángulos con tamaños muestrales decrecientes por periodo de desarrollo.

## Sensibilidad a colas más pesadas

Como extensión prevista en la metodología, se repite el experimento bajo una distribución Lognormal. Con el fin de mantener un tiempo de ejecución razonable dentro del cuaderno, esta sensibilidad se evalúa con 400 réplicas por escenario. Su propósito es verificar si el orden relativo entre métodos cambia cuando la generación de incrementales presenta colas más pesadas.

In [ ]:
# Experimento complementario con distribución Lognormal
N_REPLICAS_LOGNORMAL = 400
config_lognormal = clone_config(config, distribution="lognormal", random_seed=SEED + 1)
results_lognormal = run_experiment(config_lognormal, scenarios, n_replicas=N_REPLICAS_LOGNORMAL)
metrics_lognormal = compute_method_metrics(results_lognormal)
ranking_lognormal = rank_methods_within_scenario(metrics_lognormal, metric="rmse")
global_summary_lognormal = build_global_summary(metrics_lognormal)
dominance_lognormal = summarize_method_dominance(ranking_lognormal)

print("Resumen global bajo Lognormal:")
display(global_summary_lognormal)
print("Dominancia por escenarios bajo Lognormal:")
display(dominance_lognormal)

In [ ]:
# Comparación resumida entre resultados Gamma y Lognormal
gamma_vs_lognormal = (
    global_summary.loc[:, ["method", "mean_rmse", "mean_mape"]]
    .rename(columns={"mean_rmse": "gamma_mean_rmse", "mean_mape": "gamma_mean_mape"})
    .merge(
        global_summary_lognormal.loc[:, ["method", "mean_rmse", "mean_mape"]]
        .rename(columns={"mean_rmse": "lognormal_mean_rmse", "mean_mape": "lognormal_mean_mape"}),
        on="method",
        how="inner",
    )
    .sort_values("gamma_mean_rmse")
    .reset_index(drop=True)
)
gamma_vs_lognormal

La comparación anterior permite verificar si el orden relativo entre métodos cambia cuando la distribución subyacente presenta colas más pesadas. Si el método ganador se mantiene, la conclusión principal del estudio resulta más robusta; si cambia, ello indicaría que parte del hallazgo depende de la forma específica del proceso generador de datos.

## Conclusiones del experimento

El conjunto de resultados permite extraer cuatro conclusiones principales:

1. **El método clásico conserva eficiencia en datos limpios o débilmente contaminados.** Esto es consistente con la idea de que la robustificación puede implicar un costo cuando no hay observaciones extremas influyentes.
2. **La mediana ofrece la respuesta más consistente cuando la contaminación aumenta en magnitud o frecuencia.** En el diseño implementado, este método concentra la mayor cantidad de victorias por escenario y exhibe el mejor desempeño global en los escenarios contaminados.
3. **La media truncada y el método ponderado no dominan de forma general.** En particular, la media truncada no confirma la hipótesis de equilibrio robustez-eficiencia bajo la parametrización considerada, mientras que el método ponderado mejora en algunos contextos, pero no alcanza una ventaja sistemática.
4. **La conclusión principal es estable bajo la sensibilidad Lognormal.** El patrón general observado con Gamma no desaparece al introducir colas más pesadas, lo que fortalece la interpretación sustantiva del experimento.

En términos de la pregunta de investigación, la evidencia sugiere que los métodos robustos sí ofrecen mejoras frente al Chain-Ladder clásico, pero no de manera uniforme: la ventaja aparece principalmente cuando los outliers son lo suficientemente severos o frecuentes como para distorsionar los factores de desarrollo clásicos.

## Exportación de resultados

Con fines de trazabilidad y documentación del repositorio, los principales resultados se exportan a la carpeta `results`. Esto permite desacoplar la ejecución del cuaderno del uso posterior de tablas y figuras en el documento escrito.

In [ ]:
# Exportación de tablas finales para anexos o presentación
export_map = {
    "scenario_design.csv": scenario_design_df,
    "metrics_gamma_1000rep.csv": metrics_df,
    "ranking_gamma_1000rep.csv": ranking_df,
    "comparisons_vs_classical_gamma_1000rep.csv": comparisons_df,
    "comparison_summary_gamma_1000rep.csv": comparison_summary,
    "global_summary_gamma_1000rep.csv": global_summary,
    "environment_summary_gamma_1000rep.csv": environment_summary,
    "best_by_scenario_gamma_1000rep.csv": best_by_scenario,
    "key_results_gamma_1000rep.csv": key_results_df,
    "method_dominance_gamma_1000rep.csv": dominance_df,
    "hypothesis_assessment_gamma_1000rep.csv": hypothesis_table,
    "global_summary_lognormal_400rep.csv": global_summary_lognormal,
    "method_dominance_lognormal_400rep.csv": dominance_lognormal,
    "gamma_vs_lognormal_summary.csv": gamma_vs_lognormal,
}

exported = []
skipped = []
for filename, dataframe in export_map.items():
    output_path = OUTPUT_DIR / filename
    try:
        dataframe.to_csv(output_path, index=False)
        exported.append(filename)
    except PermissionError:
        skipped.append(filename)

print("Archivos exportados correctamente:")
for filename in exported:
    print("-", filename)
if skipped:
    print("Archivos no sobrescritos por encontrarse en uso:")
    for filename in skipped:
        print("-", filename)

## Limitaciones y alcance

El estudio aísla de forma deliberada el efecto de los valores atípicos sobre la estimación del IBNR. En consecuencia, no incorpora factores de cola, efectos calendario, dependencia entre años de ocurrencia, ajustes de exposición ni un modelo estocástico completo del tipo Mack o bootstrap para todas las variantes robustas. Estas omisiones son coherentes con el objetivo principal del trabajo, que consiste en comparar sensibilidad, precisión y estabilidad bajo contaminación controlada. No obstante, constituyen líneas naturales de extensión hacia un trabajo aplicado más amplio.

## Referencias

- Mack, T. (1993). *Distribution-free Calculation of the Standard Error of Chain Ladder Reserve Estimates*. ASTIN Bulletin. [Cambridge](https://www.cambridge.org/core/journals/astin-bulletin-journal-of-the-iaa/article/distributionfree-calculation-of-the-standard-error-of-chain-ladder-reserve-estimates/E8D207F9A4DCE30300A76780FE510437)
- England, P. D., y Verrall, R. J. (2002). *Stochastic Claims Reserving in General Insurance*. British Actuarial Journal. [Cambridge](https://www.cambridge.org/core/journals/british-actuarial-journal/article/stochastic-claims-reserving-in-general-insurance/60026990B6A88E8A6DDEECABD6506C65)
- Verdonck, T., Van Wouwe, M., y Dhaene, J. (2009). *A Robustification of the Chain-Ladder Method*. North American Actuarial Journal. [DOI](https://doi.org/10.1080/10920277.2009.10597555)
- Verdonck, T., y Debruyne, M. (2011). *The influence of individual claims on the chain-ladder estimates: Analysis and diagnostic tool*. Insurance: Mathematics and Economics. [ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S0167668710001071)
- Avanzi, B., Lavender, M., Taylor, G., y Wong, B. (2024). *On the impact of outliers in loss reserving*. European Actuarial Journal. [Springer](https://link.springer.com/article/10.1007/s13385-023-00356-2)
- Avanzi, B., Lavender, M., Taylor, G., y Wong, B. (2023). *Detection and treatment of outliers for multivariate robust loss reserving*. Annals of Actuarial Science. [DOI](https://doi.org/10.1017/S1748499523000155)
- `chainladder` para Python, documentación oficial de `MackChainladder`. [Read the Docs](https://chainladder-python.readthedocs.io/en/stable/library/generated/chainladder.MackChainladder.html)